Automated Military Data Collection & Dataset Preparation

Objective:
The purpose of this stage is to automate the process of gathering military strength data. Instead of manually collecting statistics for more than 140 countries, a Python-based script is used to systematically access the Global Firepower Index. The script reads a predefined list of country-specific URLs, extracts key indicators such as manpower, defense budgets, and military equipment counts, and compiles the collected information into a structured raw dataset ready for further processing and analysis. ⚙️📊

In [ ]:
# ==========================================
# STEP 1: SCRAPE BASE COUNTRY DATA
# ==========================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

BASE_URL = "https://www.globalfirepower.com/countries-listing.php"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def get_country_base_data():
    response = requests.get(BASE_URL, headers=HEADERS)
    soup = BeautifulSoup(response.text, "lxml")

    rank_blocks = soup.select("div.rankNumContainer")
    name_blocks = soup.select("div.countryName div.longFormName span")
    power_blocks = soup.select("div.pwrIndxContainer span")

    countries = []
    base_data = {}

    for rank, name, power in zip(rank_blocks, name_blocks, power_blocks):
        country = name.get_text(strip=True)
        countries.append(country)

        base_data[country] = {
            "rank": rank.get_text(strip=True),
            "power_index": power.get_text(strip=True)
        }

    return countries, base_data


# Execute Step 1
country_list, base_data = get_country_base_data()

print("Total Countries Scraped:", len(country_list))
display(pd.DataFrame.from_dict(base_data, orient="index").reset_index().rename(columns={"index":"country"}))

Total Countries Scraped: 145


,country,rank,power_index
0,United States,1,PwrIndx: 0.0741
1,Russia,2,PwrIndx: 0.0791
2,China,3,PwrIndx: 0.0919
3,India,4,PwrIndx: 0.1346
4,South Korea,5,PwrIndx: 0.1642
...,...,...,...
140,Liberia,141,PwrIndx: 3.9275
141,Suriname,142,PwrIndx: 4.0538
142,Central African Republic,143,PwrIndx: 4.2381
143,Beliz,144,PwrIndx: 4.3602


In [ ]:
# ==========================================
# STEP 2: SCRAPE ALL METRICS
# ==========================================

METRIC_URLS = {
    # Manpower
    'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel',
    'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel',
    'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary',
    'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service',
    'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'military_age_annually',

    # Air
    'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft',
    'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft',
    'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft',
    'https://www.globalfirepower.com/aircraft-total-transports.php': 'transport_aircraft',
    'https://www.globalfirepower.com/aircraft-total-trainers.php': 'trainer_aircraft',
    'https://www.globalfirepower.com/aircraft-total-special-mission.php': 'special_mission_aircraft',
    'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php': 'tanker_aircraft',
    'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'total_military_helicopters',
    'https://www.globalfirepower.com/aircraft-helicopters-attack.php': 'attack_helicopters',

    # Land
    'https://www.globalfirepower.com/armor-tanks-total.php': 'tanks',
    'https://www.globalfirepower.com/armor-apc-total.php': 'armored_fighting_vehicles',
    'https://www.globalfirepower.com/armor-self-propelled-guns-total.php': 'self_propelled_artillery',
    'https://www.globalfirepower.com/armor-towed-artillery-total.php': 'towed_artillery',
    'https://www.globalfirepower.com/armor-mlrs-total.php': 'rocket_projectors',

    # Navy
    'https://www.globalfirepower.com/navy-ships.php': 'total_naval_fleet',
    'https://www.globalfirepower.com/navy-force-by-tonnage.php': 'total_naval_fleet_tonnage_mt',
    'https://www.globalfirepower.com/navy-aircraft-carriers.php': 'aircraft_carriers',
    'https://www.globalfirepower.com/navy-helo-carriers.php': 'helicopter_carriers',
    'https://www.globalfirepower.com/navy-submarines.php': 'submarines',
    'https://www.globalfirepower.com/navy-destroyers.php': 'destroyers',
    'https://www.globalfirepower.com/navy-frigates.php': 'frigates',
    'https://www.globalfirepower.com/navy-corvettes.php': 'corvettes',
    'https://www.globalfirepower.com/navy-patrol-coastal-craft.php': 'coastal_patrol_craft',

    # Economy
    'https://www.globalfirepower.com/defense-spending-budget.php': 'defense_budget_usd',
    'https://www.globalfirepower.com/purchasing-power-parity.php': 'gdp_ppp_usd',
    'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php': 'forex_reserves_usd',
    'https://www.globalfirepower.com/external-debt-by-country.php': 'external_debt_usd',
}

def scrape_metric_page(url, save_html=False):
    try:
        response = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "lxml")

        if save_html:
            os.makedirs("debug_html", exist_ok=True)
            filename = url.split("/")[-1]
            with open(f"debug_html/{filename}", "w", encoding="utf-8") as f:
                f.write(response.text)

        country_spans = soup.select("div.countryNameContainer div.longFormName span")
        value_spans = soup.select("div.valueContainer span span")

        metric_data = {}

        for c, v in zip(country_spans, value_spans):
            country = c.get_text(strip=True)
            raw_value = v.get_text(strip=True)
            metric_data[country] = raw_value

        return metric_data, True

    except:
        return {}, False


# Execute Step 2
master_data = {country: base_data.get(country, {}) for country in country_list}

success_count = 0

for url, column_name in METRIC_URLS.items():
    print("Scraping:", column_name)

    metric_data, success = scrape_metric_page(url)

    if success:
        success_count += 1

    for country in country_list:
        master_data[country][column_name] = metric_data.get(country, "N/A")

    time.sleep(1)

print("URL Success Rate:", round((success_count/len(METRIC_URLS))*100, 2), "%")

Scraping: active_personnel
Scraping: reserve_personnel
Scraping: paramilitary
Scraping: fit_for_service
Scraping: military_age_annually
Scraping: total_military_aircraft
Scraping: fighter_aircraft
Scraping: attack_aircraft
Scraping: transport_aircraft
Scraping: trainer_aircraft
Scraping: special_mission_aircraft
Scraping: tanker_aircraft
Scraping: total_military_helicopters
Scraping: attack_helicopters
Scraping: tanks
Scraping: armored_fighting_vehicles
Scraping: self_propelled_artillery
Scraping: towed_artillery
Scraping: rocket_projectors
Scraping: total_naval_fleet
Scraping: total_naval_fleet_tonnage_mt
Scraping: aircraft_carriers
Scraping: helicopter_carriers
Scraping: submarines
Scraping: destroyers
Scraping: frigates
Scraping: corvettes
Scraping: coastal_patrol_craft
Scraping: defense_budget_usd
Scraping: gdp_ppp_usd
Scraping: forex_reserves_usd
Scraping: external_debt_usd
URL Success Rate: 100.0 %


In [ ]:

final_df = pd.DataFrame.from_dict(master_data, orient="index")
final_df.reset_index(inplace=True)
final_df.rename(columns={"index":"country"}, inplace=True)

print("Final Dataset Size:", len(final_df))
display(final_df.head())

# Save final CSV
final_df.to_csv("military_raw_data.csv", index=False)

print("military_raw_data.csv created successfully!")

Final Dataset Size: 145


,country,rank,power_index,active_personnel,reserve_personnel,paramilitary,fit_for_service,military_age_annually,total_military_aircraft,fighter_aircraft,...,helicopter_carriers,submarines,destroyers,frigates,corvettes,coastal_patrol_craft,defense_budget_usd,gdp_ppp_usd,forex_reserves_usd,external_debt_usd
0,United States,1,PwrIndx: 0.0741,"1,333,030","799,500",0,"124,816,644","4,445,524","13,032","1,791",...,9,66,83,0,27,0,"$ \t\t\t\t\t\t\t831,500...","$ \t\t\t\t\t\t\t25,676,...","$ \t\t\t\t\t\t\t910,037...","$ \t\t\t\t\t\t\t24,538,..."
1,Russia,2,PwrIndx: 0.0791,"1,320,000","2,000,000","250,000","46,189,226","1,267,387","4,237",861,...,0,66,13,12,79,70,"$ \t\t\t\t\t\t\t212,638...","$ \t\t\t\t\t\t\t6,089,0...","$ \t\t\t\t\t\t\t597,217...","$ \t\t\t\t\t\t\t226,475..."
2,China,3,PwrIndx: 0.0919,"2,035,000","510,000","625,000","626,864,169","19,810,606","3,529","1,443",...,4,61,53,46,50,150,"$ \t\t\t\t\t\t\t303,000...","$ \t\t\t\t\t\t\t33,598,...","$ \t\t\t\t\t\t\t3,456,0...","$ \t\t\t\t\t\t\t488,114..."
3,India,4,PwrIndx: 0.1346,"1,431,000","1,000,000","2,527,000","522,786,598","23,955,181","2,183",476,...,0,18,13,18,21,146,"$ \t\t\t\t\t\t\t109,000...","$ \t\t\t\t\t\t\t14,244,...","$ \t\t\t\t\t\t\t643,043...","$ \t\t\t\t\t\t\t212,728..."
4,South Korea,5,PwrIndx: 0.1642,"450,000","3,100,000","120,000","21,353,538","416,654","1,540",242,...,2,22,14,18,3,96,"$ \t\t\t\t\t\t\t44,800,...","$ \t\t\t\t\t\t\t2,607,0...","$ \t\t\t\t\t\t\t418,219...","$ \t\t\t\t\t\t\t553,871..."


military_raw_data.csv created successfully!


In [ ]:
# ==========================================
# STEP 1: STANDARDIZE COLUMN NAMES
# ==========================================

import pandas as pd
import re

def standardize_column_name(col_name):
    col_name = col_name.strip()
    col_name = col_name.lower()
    col_name = re.sub(r"[^\w\s]", "", col_name)
    col_name = re.sub(r"\s+", "_", col_name)
    return col_name

# Load raw dataset
df = pd.read_csv("military_raw_data.csv")

print("Before Standardization:")
print(df.columns)

# Apply transformation
df.columns = [standardize_column_name(col) for col in df.columns]

print("\nAfter Standardization:")
print(df.columns)

display(df.head())

Before Standardization:
Index(['country', 'rank', 'power_index', 'active_personnel',
       'reserve_personnel', 'paramilitary', 'fit_for_service',
       'military_age_annually', 'total_military_aircraft', 'fighter_aircraft',
       'attack_aircraft', 'transport_aircraft', 'trainer_aircraft',
       'special_mission_aircraft', 'tanker_aircraft',
       'total_military_helicopters', 'attack_helicopters', 'tanks',
       'armored_fighting_vehicles', 'self_propelled_artillery',
       'towed_artillery', 'rocket_projectors', 'total_naval_fleet',
       'total_naval_fleet_tonnage_mt', 'aircraft_carriers',
       'helicopter_carriers', 'submarines', 'destroyers', 'frigates',
       'corvettes', 'coastal_patrol_craft', 'defense_budget_usd',
       'gdp_ppp_usd', 'forex_reserves_usd', 'external_debt_usd'],
      dtype='object')

After Standardization:
Index(['country', 'rank', 'power_index', 'active_personnel',
       'reserve_personnel', 'paramilitary', 'fit_for_service',
       'military_ag

,country,rank,power_index,active_personnel,reserve_personnel,paramilitary,fit_for_service,military_age_annually,total_military_aircraft,fighter_aircraft,...,helicopter_carriers,submarines,destroyers,frigates,corvettes,coastal_patrol_craft,defense_budget_usd,gdp_ppp_usd,forex_reserves_usd,external_debt_usd
0,United States,1,PwrIndx: 0.0741,"1,333,030","799,500",0,"124,816,644","4,445,524","13,032","1,791",...,9,66,83,0,27,0,"$ \t\t\t\t\t\t\t831,500...","$ \t\t\t\t\t\t\t25,676,...","$ \t\t\t\t\t\t\t910,037...","$ \t\t\t\t\t\t\t24,538,..."
1,Russia,2,PwrIndx: 0.0791,"1,320,000","2,000,000","250,000","46,189,226","1,267,387","4,237",861,...,0,66,13,12,79,70,"$ \t\t\t\t\t\t\t212,638...","$ \t\t\t\t\t\t\t6,089,0...","$ \t\t\t\t\t\t\t597,217...","$ \t\t\t\t\t\t\t226,475..."
2,China,3,PwrIndx: 0.0919,"2,035,000","510,000","625,000","626,864,169","19,810,606","3,529","1,443",...,4,61,53,46,50,150,"$ \t\t\t\t\t\t\t303,000...","$ \t\t\t\t\t\t\t33,598,...","$ \t\t\t\t\t\t\t3,456,0...","$ \t\t\t\t\t\t\t488,114..."
3,India,4,PwrIndx: 0.1346,"1,431,000","1,000,000","2,527,000","522,786,598","23,955,181","2,183",476,...,0,18,13,18,21,146,"$ \t\t\t\t\t\t\t109,000...","$ \t\t\t\t\t\t\t14,244,...","$ \t\t\t\t\t\t\t643,043...","$ \t\t\t\t\t\t\t212,728..."
4,South Korea,5,PwrIndx: 0.1642,"450,000","3,100,000","120,000","21,353,538","416,654","1,540",242,...,2,22,14,18,3,96,"$ \t\t\t\t\t\t\t44,800,...","$ \t\t\t\t\t\t\t2,607,0...","$ \t\t\t\t\t\t\t418,219...","$ \t\t\t\t\t\t\t553,871..."


# Data Cleaning & Standardization

**Objective:**
The objective of this phase is to prepare the raw data for mathematical analysis. Computers cannot perform calculations on text strings containing symbols (e.g., "$500" or "50%"). This step focuses on cleaning the dataset by removing formatting characters, standardizing column names for consistency, and handling missing information so that the dataset consists entirely of usable numeric values.

In [ ]:
# ==========================================
# STEP 2: CLEAN NUMERIC VALUES
# ==========================================

def clean_numeric_value(raw_text):
    if pd.isna(raw_text):
        return None

    text = str(raw_text).strip()

    # Remove newlines and tabs
    text = text.replace("\n", "").replace("\t", "")

    # Remove dollar and commas
    text = text.replace("$", "").replace(",", "")

    multiplier = 1

    # Handle Billion / Million / Thousand
    if "B" in text:
        multiplier = 1_000_000_000
        text = text.replace("B", "")
    elif "M" in text:
        multiplier = 1_000_000
        text = text.replace("M", "")
    elif "K" in text:
        multiplier = 1_000
        text = text.replace("K", "")

    match = re.search(r"\d+\.?\d*", text)

    if match:
        return float(match.group()) * multiplier
    else:
        return None

print("Numeric cleaning function ready.")

Numeric cleaning function ready.


In [ ]:
# ==========================================
# STEP 3: APPLY CLEANING + EXPORT
# ==========================================

print("Original Dataset Shape:", df.shape)

# Apply numeric cleaning
for col in df.columns:
    if col != "country":
        df[col] = df[col].apply(clean_numeric_value)
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nCleaned Dataset Shape:", df.shape)

display(df.head())

# Save clean dataset
df.to_csv("military_clean.csv", index=False)

print("\nCleaning complete.")
print("Saved as: military_clean.csv")

Original Dataset Shape: (145, 35)

Cleaned Dataset Shape: (145, 35)


,country,rank,power_index,active_personnel,reserve_personnel,paramilitary,fit_for_service,military_age_annually,total_military_aircraft,fighter_aircraft,...,helicopter_carriers,submarines,destroyers,frigates,corvettes,coastal_patrol_craft,defense_budget_usd,gdp_ppp_usd,forex_reserves_usd,external_debt_usd
0,United States,1.0,0.0741,1333030.0,799500.0,0.0,124816644.0,4445524.0,13032.0,1791.0,...,9.0,66.0,83.0,0.0,27.0,0.0,8.315000e+11,2.567600e+13,9.100370e+11,2.453890e+13
1,Russia,2.0,0.0791,1320000.0,2000000.0,250000.0,46189226.0,1267387.0,4237.0,861.0,...,0.0,66.0,13.0,12.0,79.0,70.0,2.126383e+11,6.089000e+12,5.972170e+11,2.264758e+11
2,China,3.0,0.0919,2035000.0,510000.0,625000.0,626864169.0,19810606.0,3529.0,1443.0,...,4.0,61.0,53.0,46.0,50.0,150.0,3.030000e+11,3.359800e+13,3.456000e+12,4.881140e+11
3,India,4.0,0.1346,1431000.0,1000000.0,2527000.0,522786598.0,23955181.0,2183.0,476.0,...,0.0,18.0,13.0,18.0,21.0,146.0,1.090000e+11,1.424400e+13,6.430430e+11,2.127280e+11
4,South Korea,5.0,0.1642,450000.0,3100000.0,120000.0,21353538.0,416654.0,1540.0,242.0,...,2.0,22.0,14.0,18.0,3.0,96.0,4.480000e+10,2.607000e+12,4.182190e+11,5.538714e+11



Cleaning complete.
Saved as: military_clean.csv


In [ ]:
# ==========================================
# MISSING VALUE IMPUTATION (MEAN REPLACEMENT)
# ==========================================

import pandas as pd

# Load dataset
file_path = "military_clean.csv"
df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)

# ------------------------------------------
# Missing Values BEFORE
# ------------------------------------------
print("\nMissing Values BEFORE Imputation:")
missing_before = df.isnull().sum()
print(missing_before)

# Optional: Percentage missing
print("\nMissing Percentage BEFORE:")
print((missing_before / len(df) * 100).round(2))

# ------------------------------------------
# Mean Imputation for Numeric Columns
# ------------------------------------------
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in numeric_cols:
    mean_value = df[col].mean()
    df[col] = df[col].fillna(mean_value)

# ------------------------------------------
# Missing Values AFTER
# ------------------------------------------
print("\nMissing Values AFTER Imputation:")
missing_after = df.isnull().sum()
print(missing_after)

print("\nMissing Percentage AFTER:")
print((missing_after / len(df) * 100).round(2))

# ------------------------------------------
# Save Clean Dataset
# ------------------------------------------
df.to_csv("military_clean_final.csv", index=False)

print("\nMean replacement complete.")
print("File saved as: military_clean_final.csv")

Dataset Shape: (145, 35)

Missing Values BEFORE Imputation:
country                          0
rank                             0
power_index                      0
active_personnel                 0
reserve_personnel                0
paramilitary                     0
fit_for_service                  0
military_age_annually            0
total_military_aircraft          0
fighter_aircraft                 0
attack_aircraft                  0
transport_aircraft               0
trainer_aircraft                 0
special_mission_aircraft         0
tanker_aircraft                  0
total_military_helicopters       0
attack_helicopters               0
tanks                            0
armored_fighting_vehicles        0
self_propelled_artillery         0
towed_artillery                  0
rocket_projectors                0
total_naval_fleet                0
total_naval_fleet_tonnage_mt    94
aircraft_carriers                0
helicopter_carriers              0
submarines                    

In [ ]:
!pip install pycountry pycountry-convert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.1/254.1 kB 18.9 MB/s eta 0:00:00


In [ ]:
print(df[df['country'] == "Turkiye"][['country','country_fixed','iso_code']])

   country country_fixed iso_code
8  Turkiye        Turkey     None


In [ ]:
# ==========================================
# FINAL MILITARY DATA PIPELINE
# Metadata + ISO + Flags + KPIs + Export
# ==========================================

import pandas as pd
import numpy as np
import pycountry
import pycountry_convert as pc

# ------------------------------------------
# CONFIGURATION
# ------------------------------------------

INPUT_FILE = "military_clean_final.csv"
OUTPUT_FILE = "military_final.csv"

# ------------------------------------------
# STEP 1: LOAD DATA
# ------------------------------------------

print(f"Step 1: Loading '{INPUT_FILE}'...")

df = pd.read_csv(INPUT_FILE)
df.columns = df.columns.str.strip().str.lower()

print(f" -> Loaded {len(df)} rows.")


# ------------------------------------------
# STEP 2: COUNTRY CLEANING (NO HYPHENS)
# ------------------------------------------

print("\nStep 2: Cleaning Country Names...")

df['country'] = df['country'].str.strip()

country_corrections = {
    "Russia": "Russian Federation",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Ivory Coast": "Côte d'Ivoire",
    "Democratic Republic of the Congo": "Congo, The Democratic Republic of the",
    "Republic of the Congo": "Congo",
    "Turkiye": "Turkey",
    "Beliz": "Belize"
}

df['country_fixed'] = df['country'].replace(country_corrections)


# ------------------------------------------
# STEP 3: ISO CODE + FLAG URL
# ------------------------------------------

print("\nStep 3: Generating ISO Codes & Flags...")

def get_iso_code(country_name):

    manual_iso = {
        "Kosovo": "xk",
        "Turkey": "tr",
        "Türkiye": "tr"
    }

    if country_name in manual_iso:
        return manual_iso[country_name]

    try:
        country = pycountry.countries.lookup(country_name)
        return country.alpha_2.lower()
    except:
        return None

df['iso_code'] = df['country_fixed'].apply(get_iso_code)

df['flag_url'] = df['iso_code'].apply(
    lambda x: f"https://flagcdn.com/w320/{x}.png" if pd.notnull(x) else None
)

print("Unmapped countries:")
print(df[df['iso_code'].isnull()]['country'].tolist())


# ------------------------------------------
# STEP 4: CONTINENT + REGION
# ------------------------------------------

print("\nStep 4: Adding Continent & Region...")

continent_map = {
    'AF': 'Africa',
    'AS': 'Asia',
    'EU': 'Europe',
    'NA': 'North America',
    'SA': 'South America',
    'OC': 'Oceania',
    'AN': 'Antarctica'
}

def get_continent(country_name):

    if country_name == "Kosovo":
        return "Europe"

    try:
        country = pycountry.countries.lookup(country_name)
        continent_code = pc.country_alpha2_to_continent_code(country.alpha_2)
        return continent_map.get(continent_code)
    except:
        return "Other"

df['continent'] = df['country_fixed'].apply(get_continent)

df['region'] = df['continent']


# ------------------------------------------
# STEP 5: NATO ALLIANCE FLAG
# ------------------------------------------

nato_list = [
"Albania","Belgium","Bulgaria","Canada","Croatia","Czechia",
"Denmark","Estonia","Finland","France","Germany","Greece",
"Hungary","Iceland","Italy","Latvia","Lithuania","Luxembourg",
"Montenegro","Netherlands","North Macedonia","Norway","Poland",
"Portugal","Romania","Slovakia","Slovenia","Spain","Sweden",
"Turkey","United Kingdom","United States"
]

df['alliance_flag'] = df['country_fixed'].apply(
    lambda x: 'NATO' if x in nato_list else 'Non-Aligned'
)

print(" -> Metadata enrichment complete.")


# ------------------------------------------
# STEP 6: KPI ENGINEERING
# ------------------------------------------

print("\nStep 6: Calculating KPIs...")

num_cols = [
    'total_military_aircraft',
    'tanks',
    'total_naval_fleet',
    'active_personnel',
    'defense_budget_usd',
    'gdp_ppp_usd',
    'power_index'
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['power_index'] = pd.to_numeric(df['power_index'], errors='coerce').fillna(999)

df['total_military_hardware'] = (
    df['total_military_aircraft'] +
    df['tanks'] +
    df['total_naval_fleet']
)

df['assets_per_capita'] = (
    df['total_military_hardware'] /
    df['active_personnel'].replace(0, 1)
).round(7)

df['budget_gdp_ratio'] = (
    df['defense_budget_usd'] /
    df['gdp_ppp_usd'].replace(0, 1)
).round(5)

df['rank'] = df['power_index'].rank(
    ascending=True,
    method='min'
).astype(int)

df['power_index_rank_gap'] = df['rank'] - df['rank'].min()

print(" -> KPIs successfully calculated.")


# ------------------------------------------
# STEP 7: VALIDATION
# ------------------------------------------

print("\nStep 7: KPI Summary Statistics")
print(df[['assets_per_capita','budget_gdp_ratio']].describe())

print("\nTop 5 by Rank:")
print(df[['country','rank']].sort_values('rank').head())

print("\nValidation Sample (India):")
print(df[df['country'] == 'India'][[
    'country','assets_per_capita','budget_gdp_ratio','rank'
]].to_string(index=False))


# ------------------------------------------
# STEP 8: EXPORT
# ------------------------------------------

print(f"\nStep 8: Exporting to '{OUTPUT_FILE}'...")
df.to_csv(OUTPUT_FILE, index=False)

print(f"[SUCCESS] Final analytics file ready: {OUTPUT_FILE}")

Step 1: Loading 'military_clean_final.csv'...
 -> Loaded 145 rows.

Step 2: Cleaning Country Names...

Step 3: Generating ISO Codes & Flags...
Unmapped countries:
[]

Step 4: Adding Continent & Region...
 -> Metadata enrichment complete.

Step 6: Calculating KPIs...
 -> KPIs successfully calculated.

Step 7: KPI Summary Statistics
       assets_per_capita  budget_gdp_ratio
count         145.000000        145.000000
mean            0.006025          0.019467
std             0.004738          0.029966
min             0.000000          0.000080
25%             0.002478          0.005830
50%             0.005042          0.012140
75%             0.008000          0.023780
max             0.021958          0.308120

Top 5 by Rank:
         country  rank
0  United States     1
1         Russia     2
2          China     3
3          India     4
4    South Korea     5

Validation Sample (India):
country  assets_per_capita  budget_gdp_ratio  rank
  India             0.0045           0.00765   

## Conclusion

This notebook successfully automated the collection, cleaning, and preparation of military strength data from the Global Firepower Index. The process involved several key stages:

1.  **Data Scraping:** Initial data, including country ranks, power indices, and detailed metrics (manpower, aircraft, naval, land forces, economic indicators), was systematically extracted from the website.
2.  **Data Cleaning & Standardization:** Raw data underwent rigorous cleaning, where column names were standardized, numeric values were processed to remove non-numeric characters and handle multipliers (B, M, K), and missing values were imputed using mean replacement.
3.  **Metadata Enrichment & KPI Engineering:** Geographic metadata (ISO codes, flags, continents, regions) was added, and new Key Performance Indicators (KPIs) such as `total_military_hardware`, `assets_per_capita`, and `budget_gdp_ratio` were engineered to provide deeper insights.

The final `military_final.csv` dataset is now ready for advanced analysis and visualization, offering a clean, structured, and enriched foundation for understanding global military power dynamics.